In [1]:
#AIE 1.6 ONLY - from AIE NB
domain_name = "hpepcai-ingress.pcai.hpecic.net"
rag_name = "pdf-extractor-1748400159158"
token_loc="/etc/secrets/ezua/.auth_token"

In [ ]:
# NOT REQ
#from platfrom 
#kubectl -n ezaddon-system get secret hpecp-bootstrap-authconfig -o jsonpath={.data.OIDC_CLIENT_SECRET} | base64 -d
CLIENT_TOKEN=''

In [ ]:
# NOT REQ
# generate access token
UA_DOMAIN="ingress."+domain_name
KC_ADDR=f"keycloak.{UA_DOMAIN}"

# generate by logging in to platform with 2FA if applicable
with open("refresh-token.txt",'r') as file:
    REFRESH_TOKEN = file.read()

payload_refresh = {
    "grant_type": "refresh_token",
    "client_id": "ua",
    "refresh_token": REFRESH_TOKEN,
    "client_secret": CLIENT_TOKEN
}

results = requests.post(f"https://{KC_ADDR}/realms/UA/protocol/openid-connect/token", data=payload_refresh)

ACCESS_TOKEN = results.json()['access_token']

In [2]:
%update_token

Token successfully refreshed.


In [3]:
### alternatively Access Token from system
#kubectl -n $NAMESPACE get secret access-token -o jsonpath='{.data.AUTH_TOKEN}' | base64 -d
with open(token_loc,"r") as file:
    ACCESS_TOKEN=file.read()
    #print(ACCESS_TOKEN)

In [4]:
import requests

In [7]:
# call directly if access token is present
url = "https://rag-coordinator."+domain_name+"/v1/chat/completions"
prompt_list = [
#    "in context you are given invoice data. based on invoice number 86865 for polychemtex what is the shipping method - only provide shipping method nothing else - provide accurate information only",
#    "in context you are given invoice data. based on invoice number 86865 for polychemtex - in the same invoice provide me item, description, qty, unit, unit price, rebate, line total in table format - only provide what is asked nothing else - provide accurate information only",
#    "in context you are given invoice data. based on invoice number 86865 for polychemtex - in the same invoice provide me item, description, qty, unit, unit price, rebate, line total in table format - provide line 1 only - if there is no line 5 then say no-line - only provide what is asked nothing else - provide accurate information only",
"in context you are given invoice data. based on invoice number 86865 for polychemtex - in the same invoice provide me item, description, qty, unit, unit price, rebate, line total in table format - provide first 3 lines only - only provide what is asked nothing else - provide accurate information only"
]

In [8]:
for prompt in prompt_list:
    payload = {
    "model": "meta/llama-3.1-8b-instruct",
    "messages": [
       {
           "role": "user",
           "content": prompt
       },
    ],
    "max_tokens": 2048,
    "temperature": 0,
    "frequency_penalty": 1
   }
    headers = {
      "x-sa-name": rag_name,
      "Authorization": f"Bearer {ACCESS_TOKEN}"
  }

    response = requests.request("POST", url, verify=False, json=payload,headers=headers)
    print(" ----------- ")
    print(prompt)
    print(response)
    print(response.json()['choices'][0]['message']['content'])
    

/opt/conda/lib/python3.11/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'rag-coordinator.hpepcai-ingress.pcai.hpecic.net'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


 ----------- 
in context you are given invoice data. based on invoice number 86865 for polychemtex - in the same invoice provide me item, description, qty, unit, unit price, rebate, line total in table format - provide first 3 lines only - only provide what is asked nothing else - provide accurate information only
<Response [200]>
| Item | Description | QTY | Unit | Unit Price | Rebate | Line Total |
| --- | --- | --- | --- | --- | --- | --- |
| M0061 | Catalyst 1 liter | 10 | Ea. | 59.99 | 9.99 | 500.00 |
| M0062 | Resin | 10 | Ea. | 59.99 | 9.99 | 500.00 |
| M2001 | Deionized water | 5 | Ea. | 1.99 |  | 9.95 |
